# 6. Zigzag Conversion

**Difficulty:** Medium  
**Topics:** String  
**Language:** Python 3  
**Problem:** [LeetCode — Zigzag Conversion](https://leetcode.com/problems/zigzag-conversion/)

Rearrange a string by writing its characters down and diagonally up a zigzag, then reading the rows. Start by drawing the pattern in a rectangular grid, then remove unused cells by storing only each row's characters. Both approaches include a short walkthrough, correctness, complexity analysis, and complete LeetCode-compatible code.

Run the notebook from top to bottom; only Python's standard library is needed. Edit the inputs in **Try your own input** to experiment. Copy the final code cell for submission.

## 1. Problem

### Statement summary

Place the characters of `s`, in their original order, in a zigzag using `numRows` rows: move downward through the rows, then diagonally upward, and repeat. Return the string obtained by reading each row from left to right, starting with the top row. The drawing's empty positions are not included in the output.

### Constraints

- `1 <= len(s) <= 1000`
- `s` contains uppercase or lowercase English letters, commas `,`, and periods `.`.
- `1 <= numRows <= 1000`

### Examples

**Example 1**

```text
Input: s = "PAYPALISHIRING", numRows = 3
Output: "PAHNAPLSIIGYIR"
```

The arrangement is:

```text
P   A   H   N
A P L S I I G
Y   I   R
```

The three rows contribute `"PAHN"`, `"APLSIIG"`, and `"YIR"`.

**Example 2**

```text
Input: s = "PAYPALISHIRING", numRows = 4
Output: "PINALSIGYAHRPI"
```

```text
P     I    N
A   L S  I G
Y A   H R
P     I
```

Reading the rows gives `"PIN"`, `"ALSIG"`, `"YAHR"`, and `"PI"`.

**Example 3**

```text
Input: s = "A", numRows = 1
Output: "A"
```

With one row, the original ordering is already the required output.

## 2. Understand the task

This is an ordering problem. Every character appears exactly once in the result; punctuation and case are preserved. The zigzag drawing introduces visual gaps, but the answer contains no added spaces or separators. Python strings are immutable, so neither approach changes the input.

With `r = numRows > 1`, the row visits repeat as:

```text
0, 1, ..., r-1, r-2, ..., 1, 0, 1, ...
```

Reverse direction only at the first and last rows. Do not repeat an endpoint when turning around.

Two useful early returns are `numRows == 1` and `numRows >= len(s)`. In the latter case, all characters fit in the first downward traversal, one per row, so reading top to bottom leaves the input unchanged.

For both walkthroughs use the supplementary input `s = "ABCDEFG"`, `numRows = 3`:

```text
A   E
B D F
C   G
```

Its rows are `AE`, `BDF`, and `CG`, producing `"AEBDFCG"`. This short example includes a full upward diagonal and the beginning of another descent.

In [1]:
# Shared supplementary input for the walkthroughs.
EXAMPLE_S = "ABCDEFG"
EXAMPLE_ROWS = 3
EXAMPLE_EXPECTED = "AEBDFCG"

## 3. Approach 1 — draw a rectangular grid

### Idea

Represent the drawing directly. Allocate `numRows` rows and `len(s)` columns, initially containing empty strings. Walk down vertically and up diagonally, placing one character at each visited position. Finally, concatenate all grid rows.

The full width is a convenient safe upper bound: at most one column is advanced per character. Most cells remain unused, making this a useful but wasteful baseline.

### Algorithm

1. Return `s` immediately if there is only one row or the row count is at least the string length.
2. Allocate an `r × n` grid filled with `""`, where `r = numRows` and `n = len(s)`.
3. Start at `(row, column) = (0, 0)`, moving downward.
4. Place the next character at the current position.
5. While descending, move one row down; at the bottom, turn upward and advance one column. While ascending diagonally, move one row up and one column right; at the top, turn downward without advancing the column.
6. After placing all characters, read every grid row from left to right and concatenate the rows.

### Dry run

`s = "ABCDEFG"`, `numRows = 3`

| Visit | Character | Position used `(row, column)` | Next position | Next movement |
|---:|---|---|---|---|
| 1 | A | (0, 0) | (1, 0) | down |
| 2 | B | (1, 0) | (2, 0) | down |
| 3 | C | (2, 0) | (1, 1) | up diagonally |
| 4 | D | (1, 1) | (0, 2) | up diagonally |
| 5 | E | (0, 2) | (1, 2) | down |
| 6 | F | (1, 2) | (2, 2) | down |
| 7 | G | (2, 2) | (1, 3) | up diagonally |

Stop after placing `G`; the next position is never filled. The allocated width is seven columns, although only the first three contain characters. Joining all cells ignores empty strings and returns `"AEBDFCG"`.

### Why it works

Before each placement, the position and direction identify the next location in the required zigzag. The update rules follow exactly one downward step or one upward diagonal step, turning at the endpoints. A downward step changes row; an upward diagonal step advances column, so no occupied position is overwritten. Every input character is placed exactly once.

Within any row, later characters are placed in strictly later columns. Reading the grid in row order therefore produces precisely the specified ordering, and empty cells add no characters. The early-return cases follow directly from the one-row or first-descent layouts.

### Complexity

For the nontrivial case `1 < r < n`:

- **Worst-case time:** O(nr). Allocation initializes `nr` cells, placement takes O(n), and joining the rows scans `nr` cells even though only `n` contain characters.
- **Auxiliary space:** O(nr), for the rectangular grid, plus O(n) output construction. The grid dominates.

The early returns take O(1) additional work and return the existing string. Bounds assume ordinary Python list and string operations.

### Implementation

In [2]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        n = len(s)
        if numRows == 1 or numRows >= n:
            return s

        grid = [[""] * n for _ in range(numRows)]
        row = column = 0
        moving_down = True
        for char in s:
            grid[row][column] = char
            if moving_down:
                if row == numRows - 1:
                    moving_down = False
                    row -= 1
                    column += 1
                else:
                    row += 1
            else:
                if row == 0:
                    moving_down = True
                    row += 1
                else:
                    row -= 1
                    column += 1
        return "".join("".join(line) for line in grid)


GridSolution = Solution
print(GridSolution().convert(EXAMPLE_S, EXAMPLE_ROWS))

AEBDFCG


In [3]:
# Teaching trace, separate from the solution implementation.
def trace_grid(s, num_rows):
    n = len(s)
    if num_rows == 1 or num_rows >= n:
        print(f"Early return: {s!r}")
        return s
    grid = [[""] * n for _ in range(num_rows)]
    row = column = 0
    moving_down = True
    for visit, char in enumerate(s, 1):
        used = (row, column)
        grid[row][column] = char
        if moving_down:
            if row == num_rows - 1:
                moving_down = False
                row -= 1
                column += 1
            else:
                row += 1
        else:
            if row == 0:
                moving_down = True
                row += 1
            else:
                row -= 1
                column += 1
        movement = "down" if moving_down else "up diagonally"
        print(f"Visit {visit}: {char!r} at {used}; next=({row}, {column}), {movement}")
    row_strings = ["".join(line) for line in grid]
    result = "".join(row_strings)
    print(f"Rows={row_strings}; return {result!r}; placements={len(s)}")
    return result

assert trace_grid(EXAMPLE_S, EXAMPLE_ROWS) == EXAMPLE_EXPECTED

Visit 1: 'A' at (0, 0); next=(1, 0), down
Visit 2: 'B' at (1, 0); next=(2, 0), down
Visit 3: 'C' at (2, 0); next=(1, 1), up diagonally
Visit 4: 'D' at (1, 1); next=(0, 2), up diagonally
Visit 5: 'E' at (0, 2); next=(1, 2), down
Visit 6: 'F' at (1, 2); next=(2, 2), down
Visit 7: 'G' at (2, 2); next=(1, 3), up diagonally
Rows=['AE', 'BDF', 'CG']; return 'AEBDFCG'; placements=7


## 4. Approach 2 — keep only row buffers

### From scanning empty cells to storing useful characters

The final answer needs each row's characters in order, not their exact columns. Replace the rectangular grid with one list per row. The row traversal is unchanged, but column tracking and all empty-cell storage disappear.

Use character lists with `append`, then join once per row. Repeatedly growing immutable row strings with `+=` can cause repeated copying; the list-based implementation makes linear construction explicit.

### Algorithm

1. Handle the same early-return cases.
2. Create one empty character list per row. Start at row `0` with `direction = 1`.
3. Append each input character to the current row's list.
4. At row `0`, set direction downward (`+1`). At row `numRows - 1`, set it upward (`-1`). At interior rows, keep it unchanged.
5. Move to `row + direction` for the next character.
6. Join each row's characters, then join the rows from top to bottom.

Direction is updated **before** moving to the next row, so the next position stays valid. The one-row guard prevents the otherwise ambiguous top-and-bottom row case.

### Dry run

`s = "ABCDEFG"`, `numRows = 3`

| Visit | Character | Row used | Direction for next move | Row buffers after append |
|---:|---|---:|---|---|
| 1 | A | 0 | down | `[A, "", ""]` |
| 2 | B | 1 | down | `[A, B, ""]` |
| 3 | C | 2 | up | `[A, B, C]` |
| 4 | D | 1 | up | `[A, BD, C]` |
| 5 | E | 0 | down | `[AE, BD, C]` |
| 6 | F | 1 | down | `[AE, BDF, C]` |
| 7 | G | 2 | up | `[AE, BDF, CG]` |

Stop after seven appends. Joining the row buffers returns `"AEBDFCG"`; no grid columns are stored or scanned.

### Why it works

Before every iteration, the current row is the row where the next character belongs in the zigzag, and each buffer contains exactly the previously processed characters assigned to that row, in input order.

Appending the next character preserves its row's order. The direction changes reproduce the zigzag's turns at the top and bottom, so moving afterward selects the correct row for the next iteration. This establishes the invariant initially at row `0` and preserves it for every character.

Characters assigned to a row appear left to right in input order in the zigzag drawing. Thus each buffer matches that row's nonempty cells from the grid approach. Joining the buffers in row order yields exactly the required result.

### Complexity

For `1 < r < n`:

- **Worst-case time:** O(n + r), which is O(n) because `r < n`. There are `r` buffer initializations, `n` amortized O(1) appends, and joins over `n` characters plus `r` row containers.
- **Auxiliary space:** O(n + r), hence O(n), for stored characters and row containers. Excluding the returned string does not make this constant-space: the buffers still hold all `n` characters, and joining may temporarily retain row strings too.

The early returns use O(1) additional work and storage. Teaching traces make snapshots of buffers for readability; their overhead is separate from this submission implementation.

### Implementation

In [4]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s

        rows = [[] for _ in range(numRows)]
        row = 0
        direction = 1
        for char in s:
            rows[row].append(char)
            if row == 0:
                direction = 1
            elif row == numRows - 1:
                direction = -1
            row += direction
        return "".join("".join(parts) for parts in rows)


RowBufferSolution = Solution
print(RowBufferSolution().convert(EXAMPLE_S, EXAMPLE_ROWS))

AEBDFCG


In [5]:
def trace_rows(s, num_rows):
    if num_rows == 1 or num_rows >= len(s):
        print(f"Early return: {s!r}")
        return s
    rows = [[] for _ in range(num_rows)]
    row = 0
    direction = 1
    for visit, char in enumerate(s, 1):
        used_row = row
        rows[row].append(char)
        if row == 0:
            direction = 1
        elif row == num_rows - 1:
            direction = -1
        snapshot = ["".join(parts) for parts in rows]
        print(f"Visit {visit}: {char!r} to row {used_row}; direction={direction:+d}; rows={snapshot}")
        row += direction
    result = "".join("".join(parts) for parts in rows)
    print(f"Return {result!r}; appends={len(s)}")
    return result

assert trace_rows(EXAMPLE_S, EXAMPLE_ROWS) == EXAMPLE_EXPECTED

Visit 1: 'A' to row 0; direction=+1; rows=['A', '', '']
Visit 2: 'B' to row 1; direction=+1; rows=['A', 'B', '']
Visit 3: 'C' to row 2; direction=-1; rows=['A', 'B', 'C']
Visit 4: 'D' to row 1; direction=-1; rows=['A', 'BD', 'C']
Visit 5: 'E' to row 0; direction=+1; rows=['AE', 'BD', 'C']
Visit 6: 'F' to row 1; direction=+1; rows=['AE', 'BDF', 'C']
Visit 7: 'G' to row 2; direction=-1; rows=['AE', 'BDF', 'CG']
Return 'AEBDFCG'; appends=7


## 5. Comparison and common mistakes

| Approach | Worst-case time in the nontrivial case | Auxiliary space | Main trade-off |
|---|---|---|---|
| Full rectangular grid | O(nr) | O(nr) | Recreates the drawing but scans many empty cells |
| Compact row buffers | O(n + r) = O(n) | O(n + r) = O(n) | Stores only characters needed for the answer |

Choose row buffers for submission. Both methods return early when `r == 1` or `r >= n`; the table describes `1 < r < n`. Direct cycle-index traversal is another way to read rows without drawing a grid, but the row-buffer method keeps the direction changes easy to follow.

Common mistakes:

- Reading column by column rather than row by row.
- Adding spaces from the drawing to the output.
- Repeating the top or bottom character when changing direction.
- Updating direction after taking a step outside the valid row range.
- Missing the one-row guard.
- Dropping punctuation or changing letter case.
- Claiming constant auxiliary space while retaining every character in row buffers.

The submission methods assume valid platform input. The manual harness checks types, bounds, and allowed characters separately; JSON parsing, validation, and teaching traces are not part of the algorithm's stated costs.

## 6. Test the solutions

Run both implementations on all official examples and supplementary cases covering two rows, a partial cycle, punctuation, case, one row, equal or excess row count, and maximum input length. Zigzag conversion has a deterministic output, so compare exact strings and also verify that every character's multiplicity is preserved.

The maximum-length repeated-character case uses 500 rows, exercising the nontrivial paths. Another case reaches the maximum row count and uses the early return.

In [6]:
from collections import Counter

TEST_CASES = [
    ("Problem example 1", "PAYPALISHIRING", 3, "PAHNAPLSIIGYIR"),
    ("Problem example 2", "PAYPALISHIRING", 4, "PINALSIGYAHRPI"),
    ("Problem example 3", "A", 1, "A"),
    ("Supplementary: short walkthrough", "ABCDEFG", 3, "AEBDFCG"),
    ("Supplementary: two rows", "ABCD", 2, "ACBD"),
    ("Supplementary: partial cycle", "ABCDE", 3, "AEBDC"),
    ("Supplementary: punctuation", "A,B.C", 3, "AC,.B"),
    ("Supplementary: case preserved", "aAbB", 2, "abAB"),
    ("Supplementary: one row", "ABC", 1, "ABC"),
    ("Supplementary: equal rows", "ABC", 3, "ABC"),
    ("Supplementary: excess rows", "AB", 5, "AB"),
    ("Supplementary: maximum length", "A" * 1000, 500, "A" * 1000),
    ("Supplementary: maximum rows", "AB", 1000, "AB"),
]
for name, s, num_rows, expected in TEST_CASES:
    for implementation in (GridSolution, RowBufferSolution):
        result = implementation().convert(s, num_rows)
        assert type(result) is str and result == expected, (name, result, expected)
        assert len(result) == len(s) and Counter(result) == Counter(s)
    shown = repr(expected) if len(expected) <= 60 else f"<{len(expected)} characters>"
    print(f"PASS | {name}: {shown}")

PASS | Problem example 1: 'PAHNAPLSIIGYIR'
PASS | Problem example 2: 'PINALSIGYAHRPI'
PASS | Problem example 3: 'A'
PASS | Supplementary: short walkthrough: 'AEBDFCG'
PASS | Supplementary: two rows: 'ACBD'
PASS | Supplementary: partial cycle: 'AEBDC'
PASS | Supplementary: punctuation: 'AC,.B'
PASS | Supplementary: case preserved: 'abAB'
PASS | Supplementary: one row: 'ABC'
PASS | Supplementary: equal rows: 'ABC'
PASS | Supplementary: excess rows: 'AB'
PASS | Supplementary: maximum length: <1000 characters>
PASS | Supplementary: maximum rows: 'AB'


## 7. Try your own input

Edit `s_text`, `num_rows_text`, and optionally `approach`, then rerun the next cell. `s_text` contains a JSON string including its double quotes, such as `'"PAYPALISHIRING"'`. Use `"rows"` for compact row buffers or `"grid"` for the full drawing simulation.

The harness checks lengths, the row-count bounds, and the allowed English letters, commas, and periods. It rejects booleans or decimal row counts rather than silently treating them as integers. Safe JSON parsing and explanatory errors keep **Run All** noninteractive. Validation scans the input in O(n) time; its parsing and allocations are separate from submission complexity.

In [7]:
import json

def run_manual(s_text, num_rows_text, approach="rows"):
    try:
        s = json.loads(s_text)
        num_rows = json.loads(num_rows_text)
        if not isinstance(s, str) or not 1 <= len(s) <= 1000:
            raise ValueError("s must be a JSON string with 1 to 1000 characters.")
        if any(not ("A" <= char <= "Z" or "a" <= char <= "z" or char in ",.") for char in s):
            raise ValueError("Use only English letters, commas, and periods in s.")
        if type(num_rows) is not int or not 1 <= num_rows <= 1000:
            raise ValueError("numRows must be an integer from 1 to 1000.")
        implementations = {"rows": RowBufferSolution, "grid": GridSolution}
        if approach not in implementations:
            raise ValueError("approach must be 'rows' or 'grid'.")
        result = implementations[approach]().convert(s, num_rows)
        preview = repr(result) if len(result) <= 60 else repr(result[:57] + "...")
        print(f"{approach}: numRows={num_rows}; result={preview}; length={len(result)}")
        return result
    except (ValueError, TypeError) as error:
        print(f"Input error: {error}")
        return None

# EDIT THESE VALUES, then run this cell again.
s_text = '"PAYPALISHIRING"'
num_rows_text = "3"
approach = "rows"  # Or "grid".
manual_result = run_manual(s_text, num_rows_text, approach)

rows: numRows=3; result='PAHNAPLSIIGYIR'; length=14


In [8]:
assert run_manual('"ABCDEFG"', "3", "grid") == "AEBDFCG"
assert run_manual('"A"', "1000") == "A"
assert run_manual('""', "2") is None
assert run_manual('["A"]', "2") is None
assert run_manual('"A B"', "2") is None
assert run_manual('"A1"', "2") is None
assert run_manual(json.dumps("é"), "2") is None
assert run_manual('not JSON', "2") is None
assert run_manual('"ABC"', "true") is None
assert run_manual('"ABC"', "2.0") is None
assert run_manual('"ABC"', "0") is None
assert run_manual('"ABC"', "1001") is None
assert run_manual(json.dumps("A" * 1001), "2") is None
assert run_manual('"ABC"', "2", "unknown") is None
print("Manual-input harness checks passed.")

grid: numRows=3; result='AEBDFCG'; length=7
rows: numRows=1000; result='A'; length=1
Input error: s must be a JSON string with 1 to 1000 characters.
Input error: s must be a JSON string with 1 to 1000 characters.
Input error: Use only English letters, commas, and periods in s.
Input error: Use only English letters, commas, and periods in s.
Input error: Use only English letters, commas, and periods in s.
Input error: Expecting value: line 1 column 1 (char 0)
Input error: numRows must be an integer from 1 to 1000.
Input error: numRows must be an integer from 1 to 1000.
Input error: numRows must be an integer from 1 to 1000.
Input error: numRows must be an integer from 1 to 1000.
Input error: s must be a JSON string with 1 to 1000 characters.
Input error: approach must be 'rows' or 'grid'.
Manual-input harness checks passed.


## 8. Final submission

Copy the next cell into LeetCode's Python 3 editor. It uses the verified `Solution.convert(self, s: str, numRows: int) -> str` interface and the compact row-buffer approach. No imports or notebook helpers are needed.

In [9]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s

        rows = [[] for _ in range(numRows)]
        row = 0
        direction = 1
        for char in s:
            rows[row].append(char)
            if row == 0:
                direction = 1
            elif row == numRows - 1:
                direction = -1
            row += direction
        return "".join("".join(parts) for parts in rows)